In [3]:
import pandas as pd
import requests
from io import StringIO
import re
import time
import numpy as np 
import matplotlib.pyplot as plt

### Problem-1:

You are given a SQL file link: https://drive.google.com/file/d/1WFt7B84LTHhMueoKmz8W-PRo7xXqmZf3/view?usp=share_link. Read the data by using the file and store it in a excel file. In this data, there are 3 tables named "invoices", "order_leads" and "sales_sql". So create 3 sheets to your excel file.

**Sorry, the file you have requested does not exist.**

### Problem-2

Go to the site: https://rapidapi.com/wirefreethought/api/geodb-cities. From here, you have to grab the API and have to choose proper routes to get the cities of different countries. After getting the right API, hit that API and create a dataframe of all the cities that you can get by using the API. Then store the dataframe to a SQL. If you need to create an account or have to subscribe, then do that (it has free subscription but has some limitations. Use that free subscription and modify your accordingly to get all the data).  

In [ ]:
### code here

API_KEY = "1996b75138mshb955cfcc4e3e266p1746c6jsnc4ca1bb51c30"

url = "https://wft-geo-db.p.rapidapi.com/v1/geo/cities"

headers = {
    "X-RapidAPI-Key": API_KEY,
    "X-RapidAPI-Host": "wft-geo-db.p.rapidapi.com"
}

all_cities = []
limit = 10
offset = 0

while True:
    querystring = {
        "limit": limit,
        "offset": offset
    }

    response = requests.get(url, headers=headers, params=querystring)
    data = response.json()
    cities = data.get("data", [])

    if not cities:
        break

    all_cities.extend(cities)
    offset += limit
    time.sleep(1)

df = pd.DataFrame(all_cities)

conn = sqlite3.connect("cities.db")
df.to_sql("cities", conn, if_exists="replace", index=False)
conn.close()

### Problem 3:

Go to this url: https://www.flipkart.com/search?q=smartphones. This is the url to find phones in flipkart website. You have to extract the below things:
1. image url of the phone
2. name of the image
3. average ratings
4. total ratings
5. total reviews
6. discounted price
7. actual price

Extract all the phones which are available in this website. So you have to use the pagination concept. **Also after requesting every page through the url, please wait for a while (minimum 2-3 seconds), otherwise your IP address can be banned to access the flipkart website later.**

After collecting all the data, save that in a JSON file.

In [5]:
# code here

import requests
from bs4 import BeautifulSoup
import json
import time

base_url = "https://www.flipkart.com/search?q=smartphones&page={}"

headers = {
    "User-Agent": "Mozilla/5.0"
}

all_data = []
page = 1

while True:
    url = base_url.format(page)
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.text, "html.parser")

    products = soup.find_all("div", {"class": "_1AtVbE"})

    page_data = []

    for product in products:
        name = product.find("div", {"class": "_4rR01T"})
        price = product.find("div", {"class": "_30jeq3 _1_WHN1"})
        actual_price = product.find("div", {"class": "_3I9_wc _27UcVY"})
        rating = product.find("div", {"class": "_3LWZlK"})
        total_ratings = product.find("span", {"class": "_2_R_DZ"})
        image = product.find("img", {"class": "_396cs4"})

        if name and price:
            page_data.append({
                "name": name.text,
                "image_url": image["src"] if image else None,
                "average_rating": rating.text if rating else None,
                "ratings_reviews": total_ratings.text if total_ratings else None,
                "discounted_price": price.text,
                "actual_price": actual_price.text if actual_price else None
            })

    if not page_data:
        break

    all_data.extend(page_data)
    print(f"Scraped page {page}")

    page += 1
    time.sleep(3)

with open("flipkart_smartphones.json", "w", encoding="utf-8") as f:
    json.dump(all_data, f, ensure_ascii=False, indent=4)